# Visualizador de gradiente 3D

## 1. Introdução
Na matemática, dada uma função diferenciável $f(x,y)$, o gradiente é o vetor formado por suas derivadas parciais: 

$$\nabla f(x, y) = \left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right)$$

Geometricamente, esse vetor aponta na direção em que $f$ cresce mais rapidamente a partir do ponto considerado, e seu módulo $|\nabla f| = \sqrt{f_x^2 + f_y^2}$, mede a taxa de crescimento nessa direção. Em qualquer ponto, o gradiente é ortogonal à curva de nível que passa por esse ponto, por isso, ao projetá-lo sobre a superfície tridimensional, ele indica visualmente o caminho de subida mais inclinado.

## 2. Objetivos 
Este código tem por objetivo construir uma ferramenta que, a partir de uma função real de duas variáveis $f(x,y)$ e de um ponto $(x, y)$ do domínio, seja capaz de:
- Calcular simbolicamente as derivadas parciais de $x$ e $y$ por meio da biblioteca `Sympy`
- Exibir o vetor gradiente em dois pontos escolhidos pelo usuário
- Determinar o módulo do gradiente no ponto analisado
- Produzir uma vizualisação tridimensional da superfície $z = f(x,y)$, com o ponto analisado destacado e o vetor gradiente projetado sobre ela.

Para isso, necessita-se das bibliotecas `Numpy` para operações numéricas e para geração da malha, `Sympy` para diferenciação simbólica e conversão numérica por meio do módulo `lambdify`. Além disso, para parte gráfica, é empregado o `Matplotlib` para a renderização da superfície bem como do vetor com os módulos `plot_surface` e `quiver`

## 3. Código

#### 3.1 Importando as bibliotecas

In [1]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

#### 3.2 Transformando x e y em símbolos matemáticos 
A partir do módulo ´`symbols` da biblioteca `Sympy` transforma-se x e y em incógnitas matemáticas ao invés de strings.

In [2]:
x, y = sp.symbols('x y')

#### 3.3 Definindo função que irá calcular o gradiente
A função `calcular_gradiente` recebe a fórmula de $f(x, y)$ como texto, converte essa string em uma expressão simbólica com `sp.sympify` e deriva parcialmente em relação a $x$ e a $y$ com `sp.diff`. Em seguida, transforma tanto a função original quanto suas derivadas em funções numéricas com `lambdify`, para que possam ser avaliadas pelo NumPy. Após isso, mostra ao usuário qual a função escolhida e quais são suas derivadas parciais. Por fim, retorna os valores numéricos de $f(x, y)$, da derivada parcial de x e da derivada parcial de y. 

In [3]:
def calcular_gradiente(formula): 
    try:
        funcao = sp.sympify(formula) 
    except:
        print("Função inválida!") #Apresenta erro caso o posto em "formula" não for possivel de ser transformado em expressão matemática
        exit()

    gx_derivada = sp.diff(funcao, x) 
    gy_derivada= sp.diff(funcao, y) 

    print(f"\nf(x, y) = {funcao}") 
    print(f"∇f(x, y) = ({gx_derivada}, {gy_derivada})")

    f = sp.lambdify((x, y), funcao, "numpy")   
    gx = sp.lambdify((x, y), gx_derivada, "numpy")  
    gy = sp.lambdify((x, y), gy_derivada, "numpy")

    return f, gx, gy 

#### 3.4 Definindo função para calcular módulo do vetor gradiente
A função `modulo_gradiente` calcula, a partir das derivadas parciais numericas de um ponto $(x,y)$, a norma de $(f_x, f_y)$, que representa o módulo do gradiente nesses pontos. 

In [4]:
def modulo_gradiente(gx, gy, px, py): 
    return np.sqrt(gx(px, py)**2 + gy(px, py)**2) 

#### 3.5 Recepção da fórmula e dos pontos a serem anaisados
Parte interativa do código, onde os usuários definirão qual das funções disponíveis será utilizada, bem como poderão definir os ponto x e y a serem avaliados.

In [ ]:
funcoes_permitidas = [
    "x + y",
    "x - y",
    "x**2 + y**2",
    "x**2 - y**2",
    "x**3 + y**3",
    "x**3 - y**3",
]

print("""
Escolha uma das funções abaixo:
  f(x,y) = x + y
  f(x,y) = x - y
  f(x,y) = x**2 + y**2
  f(x,y) = x**2 - y**2
  f(x,y) = x**3 + y**3
  f(x,y) = x**3 - y**3

Escreva exatamente como representado acima.
""")

while True:
    formula = input("Digite a função f(x, y): ")
    if formula in funcoes_permitidas:
        break
    print("Erro: função não reconhecida. Tente novamente.") #Caso a função posta pelo usuário não esteja na lista das permitidas, será apresentada a mensagem de erro. 

while True:
    try:
        px = float(input("Digite o valor de x: "))
        py = float(input("Digite o valor de y: "))
        break
    except ValueError:
        print("Os valores de x e y devem ser números. Tente novamente.")


Escolha uma das funções abaixo:
  f(x,y) = x + y
  f(x,y) = x - y
  f(x,y) = x**2 + y**2
  f(x,y) = x**2 - y**2
  f(x,y) = x**3 + y**3
  f(x,y) = x**3 - y**3

Escreva exatamente como representado acima.



#### 3.6 Execução dos cálculos e exibição dos resultados numéricos

In [ ]:
f, gx, gy = calcular_gradiente(formula)

modulo = modulo_gradiente(gx, gy, px, py)

print(f"\nPonto analisado: ({px}, {py})")
print(f"|∇f({px}, {py})| = {modulo:.4f}")

#### 3.7 Construção da grade 3D

`np.linspace` gera 100 valores igualmente espaçados no intervalo $[-4, 4]$; `np.meshgrid` combina todos os pares possíveis, formando a malha sobre a qual $Z = f(X, Y)$ é calculado. O ponto analisado é marcado em vermelho com `ax.scatter`, e o vetor gradiente projetado sobre a superfície é desenhado em preto com `ax.quiver`.

In [ ]:
lin = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(lin, lin)
Z = f(X, Y)

z_ponto = f(px, py)
gx_val  = gx(px, py)
gy_val  = gy(px, py)

fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')

ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.75)

ax.scatter(px, py, z_ponto,
           color='red', s=80,
           label=f'Ponto ({px}, {py})  |∇f| = {modulo:.4f}')

ax.quiver(px, py, z_ponto,
          gx_val, gy_val, 0,
          length=2, normalize=True, color='black',
          label='Vetor gradiente')

ax.set_title("Superfície e Vetor Gradiente")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("f(x, y)")
ax.legend()

plt.tight_layout()
plt.show()

#### 3.8 Parte 2 do código: Modo de entrada livre
No código anterior, as funções são restritas a um menu pré definido, o que garante maior controle mas limita a exploração da ferramenta. Nesta versão do código, há maior liberdade quanto às opções: o usuário pode digitar qualquer expressão matemática com as variáveis x e/ou y, ampliando o leque de funções analisáveis, como trigonométricas, raízes, entre outras. 

É válido lembrar que, mesmo sendo mais livre, ainda apresenta algumas restrições matemáticas básicas:

- **Domínio do logarítimo:** expressões do tipo $\log$ exigem $(x, y) > 0$, uma vez que o logaritmo não está definido para argumentos nulos ou negativos
- **Domínio da raiz quadrada:** expressões da forma $\sqrt(x,y)$ requerem $(x, y) \geq 0$, pois raízes de números negativos resultam em valores complexos, incompatíveis com a visualização real
- **Divisão por zero:** denominadores que se anulem em $(x, y)$ tornam a função indefinida naquele ponto
- **Resultado complexo ou indefinido:** como supracitado, um valor complexo ou indefinido estaria fora da visualização real 
- **Apenas variáveis $x$ e $y$:** a expressão não pode conter outros símbolos (como $z$, $a$, ou nomes inválidos), pois seria interpretado pelo `SymPy` como variável livre e causaria erros na avaliação numérica.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


x, y = sp.symbols('x y') 

def calcular_gradiente(formula): 
    try:
        funcao = sp.sympify(formula) 
    except SympifyError:
        print("Função inválida!") 
        exit()
        
    gx_derivada = sp.diff(funcao, x) 
    gy_derivada= sp.diff(funcao, y) 

    print(f"\nf(x, y) = {funcao}") 
    print(f"∇f(x, y) = ({gx_derivada}, {gy_derivada})") #

    f = sp.lambdify((x, y), funcao, "numpy") 
    gx = sp.lambdify((x, y), gx_derivada, "numpy")  
    gy = sp.lambdify((x, y), gy_derivada, "numpy")
    return f, gx, gy



formula = input("Digite a função f(x, y): ")
formula = sp.sympify(formula.lower())


while True:
    try:
        px = float(input("Digite o valor de x: "))
        py = float(input("Digite o valor de y: "))
        break

    except ValueError:
        print("Os valores de x e y devem ser números.")
        print("Tente novamente.")

def modulo_gradiente(gx, gy, px, py): 
    return np.sqrt(gx(px, py)**2 + gy(px, py)**2) 


f, gx, gy = calcular_gradiente(formula) 


modulo = modulo_gradiente(gx, gy, px, py)

print(f"\nPonto analisado: ({px}, {py})")
print(f"|∇f({px}, {py})| = {modulo:.4f}")


lin = np.linspace(-4, 4, 100) 
X, Y = np.meshgrid(lin, lin) 
Z = f(X, Y) 


z_ponto = f(px, py) 
gx_val = gx(px, py) 
gy_val = gy(px, py)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.75)

ax.scatter(px, py, z_ponto, color='red', s=80, label=f'|∇f| = {modulo:.4f}')
ax.quiver(px, py, z_ponto, gx_val, gy_val, 0, length=2, normalize=True, color='black')

ax.set_title("Superfície e Vetor Gradiente")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("f(x,y)")
ax.legend()

plt.show()

A principal diferença entre esta versão do código e a versão anterior consiste na remoção da lista que continha as funções matemáticas previamente permitidas para entrada do usuário. Além disso, a adição do módulo `.lower()` a fim de converter automaticamente todos os caracteres da entrada para letras minúsculas. Essa modificação aumenta a robustez do programa ao evitar erros causados por diferenças entre letras maiúsculas e minúsculas na escrita das funções matemáticas. Dessa forma, expressões como SIN(x), Sin(x) e sin(x) passam a ser interpretadas de maneira equivalente.

## 4. Resultado
A ferramenta desenvolvida é capaz de, a partir de qualquer função pertencente ao conjunto de entradas válidas, realizar automaticamente os seguintes passos:

**Derivar simbolicamente.** O `SymPy` deriva $f(x, y)$ em relação a $x$ e $y$ de forma exata, exibindo ao usuário a expressão analítica completa do gradiente $\nabla f(x, y) = (f_x, f_y)$.

**Avaliação numérica.** Por meio de `lambdify`, as expressões simbólicas são convertidas em funções NumPy, permitindo a avaliação eficiente tanto no ponto específico $(x, y)$ quanto na malha toda para a superfície.

**Cálculo do módulo.** O módulo $|\nabla f(x, y)| = \sqrt{f_x^2 + f_y^2}$ é exibido com quatro casas decimais, quantificando a taxa de crescimento máxima da função naquele ponto.

**Visualização 3D.** A superfície $z = f(x, y)$ é renderizada sobre uma malha $100 \times 100$ no domínio $[-4, 4]^2$; o ponto analisado aparece destacado em vermelho, e o vetor gradiente normalizado é projetado sobre a superfície em preto via `ax.quiver`, evidenciando geometricamente a direção de máximo crescimento.

O código produziu resultados coerentes em todos os casos testados. Para $f(x, y) = x^2 + y^2$ no ponto $(1, 1)$, por exemplo, o gradiente calculado foi $\nabla f(1, 1) = (2, 2)$ e seu módulo $|\nabla f| = 2\sqrt{2} \approx 2{,}8284$, em concordância com o resultado analítico esperado. 

## 5. Discussão
O desenvolvimento deste projeto exigiu a aprendizagem de novos conceitos relacionados à programação científica em Python. Um dos principais desafios foi a criação de gráficos tridimensionais, o que demandou o estudo das ferramentas de visualização da biblioteca Matplotlib. Além disso, foi necessário compreender o uso da biblioteca SymPy para transformar variáveis em símbolos matemáticos e realizar operações algébricas e diferenciais.

Outro aspecto importante foi o cálculo das derivadas parciais para obtenção do gradiente da função, utilizando a integração entre as bibliotecas SymPy e NumPy. Esse processo permitiu realizar tanto os cálculos simbólicos quanto as avaliações numéricas necessárias para a geração dos gráficos.

Por fim, tentei implementar melhorias no código, conforme apresentado na Seção 3.8.1, visando tentar deixá-lo mais flexível. Não consegui deixar totalmente sem erros, como eu desejava (aceitando qualquer função desejada e resultando em erro apenas caso de realmente não ser possivel calcular). No entanto, foi uma experiência enriquecedora, proporcionando aprendizado 

## 6. Conclusão 
O projeto atingiu os objetivos propostos: foi construída uma ferramenta interativa capaz de calcular e visualizar o gradiente de funções de duas variáveis a partir de entradas fornecidas diretamente pelo usuário. A integração entre `SymPy` (diferenciação simbólica), `NumPy` (avaliação numérica eficiente via `lambdify`) e `Matplotlib` (visualização 3D com `plot_surface` e `quiver`) demonstrou ser uma combinação poderosa para aplicações de visualização matemática em Python.

Do ponto de vista conceitual, a implementação reforçou a compreensão geométrica do gradiente: ao observar o vetor projetado sobre a superfície, torna-se imediatamente visível que ele aponta na direção de maior inclinação. Essa percepção visual complementa a definição formal e facilita a assimilação do conceito.

## Referências
MATPLOTLIB DEVELOPMENT TEAM. Plot 2D data on 3D plot. Matplotlib Documentation. Disponível em: https://br.matplotlib.net/stable/gallery/mplot3d/2dcollections3d.html#sphx-glr-gallery-mplot3d-2dcollections3d-py

THE MASTER CODER. Visualização interativa de dados 3D com Matplotlib. The Master Coder Blog, 2025. Disponível em: https://themastercoder.blogspot.com/2025/10/visualizacao-interativa-de-dados-3d-com.html

TOOLCLUSTER. 8 Python Math Libraries You Should Know — From stdlib to NumPy & SciPy. ToolCluster Blog, 2026. Disponível em: https://toolcluster.app/pt/blog/python-math-libraries/

Claude. Assistente de inteligência artificial. Desenvolvido pela Anthropic. Disponível em: https://claude.ai

GEOGEBRA. Calculadora 3D. Disponível em: https://www.geogebra.org/3d